# Dataset Preparation — IWSLT15 English ↔ Vietnamese

This notebook provides a complete, reproducible pipeline for:
- downloading the IWSLT15 EN–VI dataset,
- performing exploratory data analysis (EDA),
- applying rule-based cleaning,
- saving both raw and cleaned datasets for downstream model training.

**Notebook flow:**
- Block 1: Environment setup
- Block 2: Dataset loading and inspection
- Block 3: EDA and quality checks
- Block 4: Rule-based cleaning and persistence

## Block 1: Environment Setup

Import libraries, define paths, and configure plotting defaults for consistent analysis output.

In [ ]:
# ── Environment Setup ──────────────────────────────────────────────────────
import os
import sys
from pathlib import Path

# Set this to False if you intentionally want to run locally.
KAGGLE_ONLY = True
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if KAGGLE_ONLY and not IS_KAGGLE:
    raise RuntimeError(
        "This notebook is configured to run on Kaggle only.\n"
        "Open it on Kaggle, or set KAGGLE_ONLY=False in the first cell."
    )

WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
os.chdir(WORK_DIR)
sys.path.append(str(WORK_DIR))

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"

# Install dependencies (Kaggle)
if IS_KAGGLE:
    os.system("pip install datasets pandas matplotlib seaborn -q")

print("Environment : Kaggle" if IS_KAGGLE else "Environment : Local")
print(f"ROOT        : {ROOT}")
print(f"DATA_DIR    : {DATA_DIR}")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

# Display configuration
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid")

# Paths (DATA_DIR is set by the Environment Setup cell above)
RAW_OUTPUT_DIR = str(DATA_DIR / "processed" / "iwslt15_en_vi")
CLEAN_OUTPUT_DIR = str(DATA_DIR / "processed" / "iwslt15_cleaned")
os.makedirs(RAW_OUTPUT_DIR, exist_ok=True)
os.makedirs(CLEAN_OUTPUT_DIR, exist_ok=True)

print("Environment is ready.")
print(f"Raw output directory: {RAW_OUTPUT_DIR}")
print(f"Clean output directory: {CLEAN_OUTPUT_DIR}")

## Block 2: Load and Inspect Raw Dataset

Load the EN–VI dataset from Hugging Face, inspect split sizes, preview basic diagnostics, and persist a raw snapshot.

In [ ]:
print("Loading dataset from Hugging Face Hub...")
dataset = load_dataset("thainq107/iwslt2015-en-vi")

print("\nDataset structure:")
print(dataset)

print("\nFeature columns:")
print(dataset["train"].column_names)

print("\nSplit sizes:")
for split in ["train", "validation", "test"]:
    print(f"- {split}: {len(dataset[split]):,} samples")

# English-only diagnostic preview without printing raw multilingual text
print("\nPreview diagnostics from first 3 training records:")
for idx in range(3):
    en_text = str(dataset["train"][idx]["en"])
    vi_text = str(dataset["train"][idx]["vi"])
    print(
        f"- record {idx}: en_tokens={len(en_text.split())}, "
        f"vi_tokens={len(vi_text.split())}, ratio={len(en_text.split())/max(len(vi_text.split()),1):.3f}"
    )

# Save a raw snapshot for reproducibility
print("\nSaving raw dataset snapshot...")
dataset.save_to_disk(RAW_OUTPUT_DIR)
print(f"Raw dataset saved to: {RAW_OUTPUT_DIR}")

## Block 3: Exploratory Data Analysis (EDA)

Build a train DataFrame, check alignment, measure sentence lengths, identify anomalies, and visualize distributions.

In [ ]:
# Convert train split to DataFrame for easier EDA
df = dataset["train"].to_pandas()

# Sample-level diagnostics without printing raw multilingual text
print("EDA: sample diagnostics (first 5 pairs)")
for i in range(5):
    en_len = len(str(df["en"].iloc[i]).split())
    vi_len = len(str(df["vi"].iloc[i]).split())
    ratio = en_len / max(vi_len, 1)
    print(f"- pair {i}: en_tokens={en_len}, vi_tokens={vi_len}, ratio={ratio:.3f}")

# Length-based metrics
df["len_en"] = df["en"].apply(lambda x: len(str(x).split()))
df["len_vi"] = df["vi"].apply(lambda x: len(str(x).split()))
df["len_ratio_en_vi"] = df["len_en"] / (df["len_vi"] + 1e-8)

print("\nLength statistics:")
print(df[["len_en", "len_vi", "len_ratio_en_vi"]].describe(percentiles=[0.5, 0.9, 0.95, 0.98, 0.99]))

# Missing and empty checks
missing_counts = df[["en", "vi"]].isnull().sum()
empty_count = ((df["en"].str.strip() == "") | (df["vi"].str.strip() == "")).sum()

print("\nMissing value check:")
print(missing_counts)
print(f"Empty string pairs: {empty_count}")

# Visual diagnostics
plt.figure(figsize=(16, 5))

plt.subplot(1, 2, 1)
sns.histplot(df["len_en"], bins=60, color="steelblue", alpha=0.5, label="English")
sns.histplot(df["len_vi"], bins=60, color="tomato", alpha=0.5, label="Vietnamese")
plt.title("Sentence Length Distribution")
plt.xlabel("Number of tokens")
plt.xlim(0, 160)
plt.legend()

plt.subplot(1, 2, 2)
sns.scatterplot(data=df, x="len_en", y="len_vi", alpha=0.25, s=10)
plt.plot([0, 160], [0, 160], linestyle="--", color="black")
plt.title("EN-VI Length Correlation")
plt.xlabel("English length")
plt.ylabel("Vietnamese length")
plt.xlim(0, 160)
plt.ylim(0, 160)

plt.tight_layout()
plt.show()

## Block 4: Rule-Based Cleaning and Persistence

Apply practical filtering rules derived from EDA (empty text, HTML-like tags, extreme lengths, and severe EN–VI length mismatch), then save the cleaned dataset.

In [ ]:
# Cleaning thresholds based on EDA and training practicality
MIN_LEN = 2
MAX_LEN = 80
MAX_RATIO = 2.5


def is_valid_translation_pair(example):
    en_text = str(example["en"]).strip()
    vi_text = str(example["vi"]).strip()

    # 1) Reject empty strings
    if not en_text or not vi_text:
        return False

    en_len = len(en_text.split())
    vi_len = len(vi_text.split())

    # 2) Reject HTML/XML-like noise
    if any(tag in en_text for tag in ["<", ">"]) or any(tag in vi_text for tag in ["<", ">"]):
        return False

    # 3) Reject too short/too long sequences
    if not (MIN_LEN <= en_len <= MAX_LEN and MIN_LEN <= vi_len <= MAX_LEN):
        return False

    # 4) Reject severe length mismatch
    ratio = en_len / max(vi_len, 1)
    if ratio > MAX_RATIO or ratio < (1 / MAX_RATIO):
        return False

    return True


print("Applying cleaning filters to all splits...")
cleaned_dataset = dataset.filter(is_valid_translation_pair)

print("\nCleaning summary:")
for split in ["train", "validation", "test"]:
    original_size = len(dataset[split])
    cleaned_size = len(cleaned_dataset[split])
    removed_size = original_size - cleaned_size
    removed_pct = (removed_size / original_size * 100) if original_size else 0.0
    print(
        f"- {split}: kept {cleaned_size:,}/{original_size:,} "
        f"(removed {removed_size:,}, {removed_pct:.2f}%)"
    )

print("\nSaving cleaned dataset...")
cleaned_dataset.save_to_disk(CLEAN_OUTPUT_DIR)
print(f"Cleaned dataset saved to: {CLEAN_OUTPUT_DIR}")